# XLM-RoBERTa — `uni_subject` Annotation

**Model:** `xlm-roberta-base`  
**Variable:** `uni_subject`

## 1. Imports & config

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaTokenizerFast, XLMRobertaForSequenceClassification
from torch.optim import AdamW
from tqdm.auto import tqdm

from corex_eval import evaluate, load_inputs, load_training_data

MODEL_NAME   = "xlm-roberta-base"
MAX_LEN      = 128
BATCH_SIZE   = 32
EPOCHS       = 5
LR           = 2e-5
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR     = Path("saved_model")
print(f"Device: {DEVICE}")

## 2. Load training data

In [ ]:
train_df = load_training_data(
    task="annotation",
    variable="uni_subject",
    features=["subject_label", "uni_subject"],
)
train_df = train_df.dropna(subset=["subject_label", "uni_subject"]).reset_index(drop=True)
train_df["input_text"] = train_df["subject_label"]

label2id = {lbl: i for i, lbl in enumerate(sorted(train_df["uni_subject"].unique()))}
id2label = {i: lbl for lbl, i in label2id.items()}
train_df["label_id"] = train_df["uni_subject"].map(label2id)
print(f"{len(train_df)} training rows | {len(label2id)} unique subject codes")

## 3. Tokenise & fine-tune

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)

class SubjectDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc    = tokenizer(texts, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

train_ds     = SubjectDataset(train_df["input_text"].tolist(), train_df["label_id"].tolist())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label2id)).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_enc, batch_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        batch_enc    = {k: v.to(DEVICE) for k, v in batch_enc.items()}
        batch_labels = batch_labels.to(DEVICE)
        outputs      = model(**batch_enc, labels=batch_labels)
        loss         = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss  += loss.item()
    print(f"  Epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}")

SAVE_DIR.mkdir(exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved.")

## 4. Load test inputs & predict

In [ ]:
test_df = load_inputs(task="annotation", variable="uni_subject")
print(f"{len(test_df)} test rows")

test_texts = test_df["subject_label"].fillna("").tolist()
test_ds    = SubjectDataset(test_texts, [0]*len(test_texts))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

model.eval()
all_preds = []
with torch.no_grad():
    for batch_enc, _ in tqdm(test_loader, desc="Predicting"):
        batch_enc = {k: v.to(DEVICE) for k, v in batch_enc.items()}
        logits    = model(**batch_enc).logits
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy())

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = [id2label[i] for i in all_preds]
predictions_df.head()

## 5. Evaluate

In [ ]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="uni_subject",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/annotation/education/xlmroberta_uni_subject/config.yaml",
)
pd.Series({k: v for k, v in results.items() if k not in ("per_class", "per_country")})